# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed-6513/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Baseline Rule**: A page is worth reviewing if it had significant traffic (>100 impressions last month), AND it is either a young page (<180 days old) OR it faces high keyword competition (>0.71). 
The score is scaled by the prior impressions (so we prioritize high-traffic pages at risk).

**Reason Codes**:
- `YOUNG_AND_HIGH_COMP`: The page is less than 6 months old AND the keyword has high competition (>0.71).
- `YOUNG_VOLATILE`: The page is less than 6 months old and is at high risk of a traffic drop.
- `HIGH_COMPETITION`: The keyword has a high competition score (>0.71), making it harder to retain traffic.

In [ ]:
import os
import pandas as pd
import duckdb

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except ImportError:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    raise ValueError('HF_TOKEN not found.')

con = duckdb.connect()
con.execute('INSTALL httpfs;')
con.execute('LOAD httpfs;')
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

query_features = """
WITH content_base AS (
    SELECT
        content_hash_id as content_id,
        COALESCE(competition, 0) as competition,
        days_since_published
    FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
),
perf_feb AS (
    SELECT
        content_hash_id as content_id,
        SUM(gsc_impressions) as impressions_prior
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
    GROUP BY content_hash_id
)
SELECT 
    c.content_id, 
    c.competition, 
    c.days_since_published, 
    pf.impressions_prior
FROM content_base c
JOIN perf_feb pf ON c.content_id = pf.content_id
WHERE pf.impressions_prior > 100
"""
df = con.execute(query_features).df()
print(f"Loaded {len(df)} pages for baseline scoring.")

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import numpy as np

# Apply the rule components
df['is_young'] = (df['days_since_published'] < 180).astype(int)
df['is_high_comp'] = (df['competition'] > 0.71).astype(int)

# Calculate the score: (young OR high_comp) * impressions_prior
# We use max(is_young, is_high_comp) so that being both doesn't double the multiplier
df['score'] = df[['is_young', 'is_high_comp']].max(axis=1) * df['impressions_prior']

# Assign reason codes
def get_reason_code(row):
    if row['is_young'] and row['is_high_comp']:
        return 'YOUNG_AND_HIGH_COMP'
    elif row['is_young']:
        return 'YOUNG_VOLATILE'
    elif row['is_high_comp']:
        return 'HIGH_COMPETITION'
    else:
        return 'NONE'

df['reason_code'] = df.apply(get_reason_code, axis=1)

# Filter out pages with score 0 and rank the rest
df_queue = df[df['score'] > 0].copy()
df_queue = df_queue.sort_values(by='score', ascending=False)
df_queue['rank'] = range(1, len(df_queue) + 1)

# Write to CSV
os.makedirs('../outputs', exist_ok=True)
df_queue.to_csv('../outputs/baseline_action_score.csv', index=False)

print("Ranked queue built and saved!")
print("\nTop 20 pages:")
display(df_queue[['rank', 'content_id', 'competition', 'days_since_published', 'impressions_prior', 'score', 'reason_code']].head(20))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.